# Run pipeline and compare models across five elections
Run all cells to refresh the data, evaluate all eight candidates on 2005, 2010, 2015, 2017 and 2019, and select by mean election accuracy. Each forecast uses only earlier elections for tuning and fitting. The selected procedure is then retuned and fitted using history through 2019 for the 2024 retrospective benchmark.

This run includes many inner fits and neural ensembles and can take substantially longer than the previous single-election comparison. Progress is printed during selection. The notebook has not been executed as part of this refactor.

`train.csv` and `test.csv` remain in `TEST_TRAIN/`. This notebook's directory receives `model_accuracies.csv` (five-election summary), `model_accuracies_elections.csv`, `model_accuracies_predictions.csv`, and `model_accuracies_report.json` (search settings, fit records, seeds, durations, versions and completion status). Reruns overwrite these files; check the report status before using saved results.


In [ ]:
from pathlib import Path
import importlib
import sys

# Find this checkout from either the project or notebook directory.
working_directory = Path.cwd().resolve()
project_root = next(
    (folder for folder in (working_directory, *working_directory.parents)
     if (folder / "Run Pipeline" / "run_pipeline.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from inside the election project.")

pipeline_directory = project_root / "Run Pipeline"
# Prefer this checkout and import fresh pipeline code on every run.
for folder in (project_root, pipeline_directory, pipeline_directory / "additional_funcs"):
    folder_string = str(folder)
    if folder_string in sys.path:
        sys.path.remove(folder_string)
    sys.path.insert(0, folder_string)

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name in ("run_pipeline", "automated_model_selection") or module_name.startswith("Models."):
        sys.modules.pop(module_name, None)

from run_pipeline import run_pipeline
print(f"Running pipeline from: {pipeline_directory / 'run_pipeline.py'}")

output_directory = project_root / "Analysis and model development" / "00_pipeline"
print(f"Saving model accuracies to: {output_directory}")


In [ ]:
selection = run_pipeline(output_dir=output_directory)
final_model = selection.model
report = selection.report
model_type = selection.model_name

print("\nFive-election candidate comparison (equal election weights):")
display(report.scorecard())
print("\nIndividual outer-election scores:")
import pandas as pd
display(pd.DataFrame(report.outer_results)[[
    "candidate", "election", "accuracy", "changed_seat_accuracy",
    "changed_seat_evaluation_rows", "retained_seat_accuracy", "log_loss",
    "previous_winner_accuracy", "runtime_seconds"
]])
print(f"Selected model: {model_type}")
print(f"Report status: {report.status}")
print("Final fitted settings:", final_model.hyperparameters)
print("Final fit record:", report.final_fit)


## Retrospective 2024 benchmark
Evaluate the final fitted model on all test rows with known winners. Missing predictors do not remove seats from evaluation. The 2024 election has already informed project development, so this is a retrospective benchmark rather than an untouched final test. Confusion matrix rows show actual winning parties; columns show predicted winning parties.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

test_data = pd.read_csv(project_root / "TEST_TRAIN" / "test.csv")
# Predict all seats; only unknown outcomes are excluded from accuracy scoring.
all_test_predictions = pd.Series(final_model.predict(test_data), index=test_data.index)
test_evaluation = test_data.dropna(subset=["winner"])
if test_evaluation.empty:
    raise ValueError("Test data has no known winners to evaluate.")
test_predictions = all_test_predictions.loc[test_evaluation.index].to_numpy()
test_accuracy = accuracy_score(test_evaluation["winner"], test_predictions)
print(f"Test accuracy ({model_type}): {test_accuracy:.2%}")
print(f"Evaluated {len(test_evaluation)} of {len(test_data)} test seats; "
      f"excluded {len(test_data) - len(test_evaluation)} rows with missing winners.")

party_labels = sorted(set(test_evaluation["winner"]) | set(test_predictions))
test_confusion_matrix = confusion_matrix(
    test_evaluation["winner"], test_predictions, labels=party_labels
)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(test_confusion_matrix, display_labels=party_labels).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
ax.set_title(f"{model_type}: 2024 retrospective confusion matrix")
ax.set_xlabel("Predicted winning party")
ax.set_ylabel("Actual winning party")
fig.tight_layout()
plt.show()
